# Define mnemonics

In [1]:
import os
import glob
from collections import defaultdict

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

import lasio
from dlisio import dlis
from dlisio.common import ErrorHandler, Actions

## Go to base directory and find all data

In [9]:
# + '/maltseva' - sibirea
# + '/out_noname' - volga-ural
# + '/PEPS_SA' - australia
place_data = '/Volumes/ADATA' + '/PEPS_SA'
os.chdir(place_data)

las_files = glob.glob('**/*.las', recursive=True)
LAS_files = glob.glob('**/*.LAS', recursive=True)
dlis_files = glob.glob('**/*.dlis', recursive=True)
DLIS_files = glob.glob('**/*.DLIS', recursive=True)

## Count all mnemonics of files

In [15]:
mnemonics = defaultdict(int)
units = defaultdict(list)

In [16]:
for file in las_files + LAS_files:
    try:
        las = lasio.read(file, ignore_data=True, ignore_header_errors=True)
        for curve in las.curves[:]:
            mnemonics[curve.mnemonic] += 1

            if curve.mnemonic not in units:
                units[curve.mnemonic].append(curve.unit)
            else:
                if curve.unit not in units[curve.mnemonic]:
                    units[curve.mnemonic].append(curve.unit)
    except Exception as e:
        print(f"Skip {file}: {type(e).__name__}: {e}")

Line 84 (section ~PARAMETER INFORMATION): "------------------------------------------------------------"
Line 44 (section ~Curve Information Block): "SPERRY DRILLING SERVICES WILL USE ITS BEST EFFORTS TO FURNISH CUSTOMERS WITH"
Line 45 (section ~Curve Information Block): "ACCURATE INFORMATION AND INTERPRETATIONS THAT ARE PART OF, AND INCIDENT TO,"
Line 48 (section ~Curve Information Block): "UNDER NO CIRCUMSTANCES SHOULD ANY SUCH INFORMATION OR INTERPRETATION BE RELIED"
Line 49 (section ~Curve Information Block): "UPON AS THE SOLE BASIS FOR ANY DRILLING, COMPLETION, PRODUCTION, OR FINANCIAL"
Line 50 (section ~Curve Information Block): "DECISION OR ANY PROCEDURE INVOLVING ANY RISK TO THE SAFETY OF ANY DRILLING"
Line 53 (section ~Curve Information Block): "DRILLING SERVICES MAKES NO REPRESENTATIONS OR WARRANTIES, EITHER EXPRESSED OR"
Line 54 (section ~Curve Information Block): "IMPLIED, INCLUDING, BUT NOT LIMITED TO, THE IMPLIED WARRANTIES OF"
Line 55 (section ~Curve Information Block): 

Skip downloaded_files/Bugito 006 [4183]/bugito_6_cbl_aug24_drift_r1.las: AttributeError: 'NoneType' object has no attribute 'copy'
Skip downloaded_files/Bugito 006 [4183]/bugito_6_cbl_aug24_freepipe_r1.las: AttributeError: 'NoneType' object has no attribute 'copy'
Skip downloaded_files/Bugito 006 [4183]/bugito_6_cbl_aug24_main_r1.las: AttributeError: 'NoneType' object has no attribute 'copy'
Skip downloaded_files/Bugito 006 [4183]/bugito_6_cbl_aug24_repeat_r1.las: AttributeError: 'NoneType' object has no attribute 'copy'


In [17]:
eh = ErrorHandler(critical=Actions.LOG_ERROR)

for file in dlis_files + DLIS_files:
    try:
        with dlis.load(file, error_handler=eh) as files:
            for f in files:
                for frame in f.frames:
                    mn = [ch.name for ch in frame.channels if ch is not None]
                    uni = [ch.units for ch in frame.channels if ch is not None]
                    for m, u in zip(mn, uni):
                        mnemonics[m] += 1
                        if m not in units:
                            units[m].append(u)
                        else:
                            if u not in units[m]:
                                units[m].append(u)
    except Exception as e:
        print(f"Skip {file}: {type(e).__name__}: {e}")

Unable to find linked object: Multiple CHANNEL objects with name DEPTH. Candidates are:
origin=1, copy=0,
origin=1, copy=0,
Unable to find linked object: Multiple CHANNEL objects with name DEPTH. Candidates are:
origin=1, copy=0,
origin=1, copy=0,
Unable to find linked object: Multiple CHANNEL objects with name DEPTH. Candidates are:
origin=1, copy=0,
origin=1, copy=0,
Unable to find linked object: Multiple CHANNEL objects with name DEPTH. Candidates are:
origin=1, copy=0,
origin=1, copy=0,
Unable to find linked object: Multiple CHANNEL objects with name HRD22. Candidates are:
origin=0, copy=0,
origin=0, copy=0,
Unable to find linked object: Multiple CHANNEL objects with name HRD22. Candidates are:
origin=0, copy=0,
origin=0, copy=0,
Unable to find linked object: Multiple CHANNEL objects with name SFT22. Candidates are:
origin=0, copy=0,
origin=0, copy=0,
Unable to find linked object: Multiple CHANNEL objects with name SFT22. Candidates are:
origin=0, copy=0,
origin=0, copy=0,
Unable t

Skip downloaded_files/Boston 002 [2965]/boston_2_rcbl.dlis: RuntimeError: searched 200 bytes from offset 0 (dec), but could not find visible record envelope pattern [0xFF 0x01]



Problem:      rp66: Incorrect format version in Visible Record 1025
Where:        dlis::findoffsets (indexing logical file)
Severity:     critical
Action taken: Indexing is suspended at last valid Logical Record
Debug info:   Physical tell: 8388676 (dec), Logical Record tell: 8374640 (dec), Logical Record Segment tell: 8384496 (dec)
Unable to find linked object: Multiple CHANNEL objects with name DEPTH. Candidates are:
origin=1, copy=0,
origin=1, copy=0,
Unable to find linked object: Multiple CHANNEL objects with name DEPTH. Candidates are:
origin=1, copy=0,
origin=1, copy=0,
Unable to find linked object: Multiple CHANNEL objects with name DEPTH. Candidates are:
origin=1, copy=0,
origin=1, copy=0,
Unable to find linked object: Multiple CHANNEL objects with name DEPTH. Candidates are:
origin=1, copy=0,
origin=1, copy=0,
Unable to find linked object: Multiple CHANNEL objects with name DEPTH. Candidates are:
origin=1, copy=0,
origin=1, copy=0,
Unable to find linked object: Multiple CHANN

In [23]:
copybar = mnemonics.copy()
sorted_items = sorted(copybar.items(), key=lambda x: x[1], reverse=False)
copybar = dict(sorted_items)

total_app = 2000
to_max = len(mnemonics)
from_min = to_max - total_app

sf = 3
h = round((500 / (7477 - 4000)) * (to_max - from_min))
addh = 0 # = 10
addv = 0 #= 3000
fig = plt.figure(figsize=(30 * sf + addh, h * sf + addv))
ax = fig.add_subplot(231)

plt.title(f"Mnemonics of las and dlis files : {len(mnemonics)} curves types")
ax.barh(list(copybar.keys())[from_min:to_max], list(copybar.values())[from_min:to_max])
ax.margins(y=0)
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')
ax.set_ylabel("Mnemonics")
ax.set_xlabel("Amount")

Text(0.5, 0, 'Amount')

In [19]:
place_picture = '/Users/leonm4/Desktop/Oil/LeoOilML/Picture'
os.chdir(place_picture)

#fig.savefig("Mnemonics.png", dpi=300, bbox_inches='tight', transparent=True)

#fig.savefig("Mnemonics_romashka.png", dpi=300, bbox_inches='tight', transparent=True)
#fig.savefig("Mnemonics_sib.png", dpi=300, bbox_inches='tight', transparent=True)
fig.savefig("Mnemonics_australia.png", dpi=300, bbox_inches='tight', transparent=True)

os.chdir(place_data)

In [20]:
place_dict = '/Users/leonm4/Desktop/Oil/LeoOilML'
os.chdir(place_dict)

with open("list_synonyms.txt", "w", encoding="utf-8") as file:
    file.write(str(len(mnemonics)) + "")
    #file.writelines(mnemonics.keys())
    for mnemonic in mnemonics.keys():
        file.write(mnemonic + "|")
        file.write(str(len(units[mnemonic])) + "\t")
        for unit in units[mnemonic]:
            if isinstance(unit, bytes):
                unit = unit.decode('utf-8', errors='replace')
            if unit == '' or unit is None:
                file.write("_" + "|")
            else:
                file.write(unit + "|")
        file.write("")

os.chdir(place_data)

# Structure of $\ $ *.dlise files

In [3]:
def dlis_frame_to_dataframe(frame):
    curves = frame.curves()
    data = defaultdict(list)
    for name in curves.dtype.names:
        field = curves[name]
        field_dtype = curves.dtype[name]

        if field_dtype.shape == ():
            data[name] = field
        else:
            if field.ndim == 2:
                flat = np.array([np.array(x).flatten() for x in field])
                for i in range(flat.shape[1]):
                    data[f"{name}_{i+1}"] = flat[:, i]

    df = pd.DataFrame(data)
    return df

In [12]:
eh = ErrorHandler(critical=Actions.LOG_ERROR)
with dlis.load(dlis_files[2], error_handler=eh) as file:
    print(file.describe())
    for f in file:
        print(f.describe())
        for frame in f.frames:
            print(frame.describe())
            df = dlis_frame_to_dataframe(frame)
            print(df.head(10))

-------------
Physical File
-------------
Number of Logical Files : 1

Description : LogicalFile(CMR_047LUP)
Frames      : 2
Channels    : 297


------------
Logical File
------------
Description : LogicalFile(CMR_047LUP)
Frames      : 2
Channels    : 297

Known objects
--
TOOL                    : 4
CHANNEL                 : 297
FRAME                   : 2
CALIBRATION-MEASUREMENT : 101
PARAMETER               : 324
FILE-HEADER             : 1
CALIBRATION             : 288
PROCESS                 : 1
ORIGIN                  : 1
CALIBRATION-COEFFICIENT : 108
EQUIPMENT               : 14

Unknown objects
--
440-OP-CHANNEL               : 297
440-CHANNEL                  : 280
440-PRESENTATION-DESCRIPTION : 1


-----
Frame
-----
name   : 60B
origin : 41
copy   : 0

Channel indexing
--
Indexed by       : BOREHOLE-DEPTH
Index units      : 0.1 in
Index min        : 0 [0.1 in]
Index max        : 0 [0.1 in]
Direction        : DECREASING
Constant spacing : -60 [0.1 in]
Index channel    : Channe

In [15]:
place_dict = '/Users/leonm4/Desktop/Oil/LeoOilML'
os.chdir(place_dict)